In [3]:
import os
import pandas as pd
import numpy as np

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_FOLDER = os.path.join(BASE_DIR, "data")
OUTPUT_FOLDER = os.path.join(BASE_DIR, "outputs")

os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [4]:
df = pd.read_csv("data/IMDB Dataset.csv")

print(df.head())
print(df.shape)
print(df.columns)
print(df.info())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)
Index(['review', 'sentiment'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


In [5]:
print(df["sentiment"].value_counts())
print(df["sentiment"].value_counts(normalize=True) * 100)

sentiment
positive    25000
negative    25000
Name: count, dtype: int64
sentiment
positive    50.0
negative    50.0
Name: proportion, dtype: float64


In [6]:
print(df.isnull().sum())

review       0
sentiment    0
dtype: int64


In [7]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 418


In [8]:
df_clean = df.drop_duplicates().copy()

print("Before:", df.shape)
print("After:", df_clean.shape)

Before: (50000, 2)
After: (49582, 2)


In [9]:
cleaned_path = os.path.join(OUTPUT_FOLDER, "IMDB_Cleaned.csv")
df_clean.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved:", cleaned_path)

Cleaned dataset saved: f:\Crack_404\week4_sentiment_analysis\outputs\IMDB_Cleaned.csv


In [10]:
import re

from collections import Counter
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split

In [11]:
df = pd.read_csv("outputs/IMDB_Cleaned.csv")

print(df.head())
print("\nDataset shape:", df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset shape: (49582, 2)


In [12]:
df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})

print(df[["sentiment", "label"]].head())
print("\nLabel distribution:")
print(df["label"].value_counts())

  sentiment  label
0  positive      1
1  positive      1
2  positive      1
3  negative      0
4  positive      1

Label distribution:
label
1    24884
0    24698
Name: count, dtype: int64


In [13]:
stemmer = PorterStemmer()

stop_words = set(ENGLISH_STOP_WORDS)

important_words = {
    "not",
    "no",
    "never",
    "nor"
}

stop_words = stop_words - important_words

In [14]:
def preprocess_text(text):
    text = text.lower()

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    processed_tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words
    ]

    return processed_tokens

In [15]:
df["tokens"] = df["review"].apply(preprocess_text)

print(df[["review", "tokens"]].head())

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                              tokens  
0  [review, mention, watch, just, oz, episod, ll,...  
1  [wonder, littl, product, film, techniqu, unass...  
2  [thought, wonder, way, spend, time, hot, summe...  
3  [basic, s, famili, littl, boy, jake, think, s,...  
4  [petter, mattei, s, love, time, money, visual,...  


In [16]:
word_counts = Counter()

for tokens in df["tokens"]:
    word_counts.update(tokens)

print("Unique words:", len(word_counts))
print("\nMost common words:")
print(word_counts.most_common(20))

Unique words: 68957

Most common words:
[('s', 128912), ('movi', 102503), ('film', 95233), ('t', 67751), ('not', 60300), ('like', 44836), ('just', 34928), ('time', 31702), ('good', 29989), ('make', 29831), ('charact', 28129), ('watch', 27616), ('stori', 25121), ('no', 25099), ('realli', 22924), ('scene', 21300), ('look', 19787), ('end', 18961), ('bad', 18363), ('great', 18276)]


In [17]:
VOCAB_SIZE = 20000

most_common_words = word_counts.most_common(VOCAB_SIZE - 2)

word_to_index = {
    "<PAD>": 0,
    "<UNK>": 1
}

for index, (word, count) in enumerate(most_common_words, start=2):
    word_to_index[word] = index

print("Vocabulary size:", len(word_to_index))

Vocabulary size: 20000


In [18]:
def encode_review(tokens):
    return [
        word_to_index.get(word, word_to_index["<UNK>"])
        for word in tokens
    ]

In [19]:
df["encoded_review"] = df["tokens"].apply(encode_review)

print(df["encoded_review"].head())

0    [202, 309, 13, 8, 2557, 156, 128, 1511, 97, 50...
1    [94, 44, 170, 4, 1494, 12520, 60, 9, 1844, 908...
2    [81, 94, 28, 601, 9, 707, 1225, 2081, 382, 575...
3    [346, 2, 111, 44, 173, 2686, 25, 2, 465, 2968,...
4    [1, 7824, 2, 23, 9, 177, 459, 1017, 4, 13, 274...
Name: encoded_review, dtype: object


In [20]:
MAX_LENGTH = 200

def pad_sequence(sequence, max_length=MAX_LENGTH):
    if len(sequence) >= max_length:
        return sequence[:max_length]

    padding_length = max_length - len(sequence)

    return sequence + [0] * padding_length

In [21]:
df["padded_review"] = df["encoded_review"].apply(pad_sequence)

print("Review length:", len(df["padded_review"].iloc[0]))
print(df["padded_review"].iloc[0][:30])

Review length: 200
[202, 309, 13, 8, 2557, 156, 128, 1511, 97, 505, 93, 29, 2658, 2557, 1016, 10735, 17, 449, 73, 97, 250, 1268, 6, 4702, 289, 7992, 529, 15, 1560, 985]


In [23]:
X = np.array(df["padded_review"].tolist())
y = df["label"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (49582, 200)
y shape: (49582,)


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39665
Testing samples: 9917


In [26]:
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [27]:
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [29]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [30]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 620
Testing batches: 155


In [31]:
class SentimentModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim=128,hidden_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)

        mask = (x != 0).unsqueeze(-1)

        masked_embeddings = embedded * mask

        summed = masked_embeddings.sum(dim=1)

        lengths = mask.sum(dim=1).clamp(min=1)

        pooled = summed / lengths

        output = self.fc1(pooled)
        output = self.relu(output)
        output = self.dropout(output)
        output = self.fc2(output)
        output = self.sigmoid(output)

        return output.squeeze(1)

In [32]:
model = SentimentModel(len(word_to_index))

model = model.to(device)

print(model)

SentimentModel(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [33]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [34]:
def train_model(
    model,
    train_loader,
    criterion,
    optimizer,
    epochs=5
):
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

            predictions = (outputs >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        average_loss = total_loss / len(train_loader)
        accuracy = correct / total

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {average_loss:.4f} | "
            f"Accuracy: {accuracy:.2%}"
        )

In [35]:
EPOCHS = 5

train_model(
    model,
    train_loader,
    criterion,
    optimizer,
    EPOCHS
)

Epoch 1/5 | Loss: 0.4414 | Accuracy: 79.60%
Epoch 2/5 | Loss: 0.2752 | Accuracy: 89.07%
Epoch 3/5 | Loss: 0.2228 | Accuracy: 91.59%
Epoch 4/5 | Loss: 0.1864 | Accuracy: 93.23%
Epoch 5/5 | Loss: 0.1590 | Accuracy: 94.49%


In [37]:
def evaluate_model(model, test_loader):
    model.eval()

    correct = 0
    total = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            predictions = (outputs >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    accuracy = correct / total

    print(f"Test Accuracy: {accuracy:.2%}")

    return all_labels, all_predictions

In [38]:
test_labels, test_predictions = evaluate_model(model, test_loader)

Test Accuracy: 87.94%


In [39]:
from sklearn.metrics import classification_report, confusion_matrix

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=["Negative", "Positive"]
    )
)

cm = confusion_matrix(test_labels, test_predictions)

print("Confusion Matrix:")
print(cm)

              precision    recall  f1-score   support

    Negative       0.88      0.88      0.88      4940
    Positive       0.88      0.88      0.88      4977

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917

Confusion Matrix:
[[4350  590]
 [ 606 4371]]


In [40]:
import os

os.makedirs("models", exist_ok=True)

MODEL_PATH = "models/sentiment_model.pt"

torch.save(model.state_dict(), MODEL_PATH)

print("Model saved to:", MODEL_PATH)

Model saved to: models/sentiment_model.pt


In [41]:
import json

VOCAB_PATH = "models/vocabulary.json"

with open(VOCAB_PATH, "w", encoding="utf-8") as file:
    json.dump(word_to_index, file)

print("Vocabulary saved to:", VOCAB_PATH)

Vocabulary saved to: models/vocabulary.json


In [42]:
def prepare_sentence(sentence):
    tokens = preprocess_text(sentence)

    encoded = [
        word_to_index.get(word, word_to_index["<UNK>"])
        for word in tokens
    ]

    padded = pad_sequence(encoded)

    tensor = torch.tensor(
        [padded],
        dtype=torch.long
    )

    return tensor.to(device)

In [43]:
def predict_sentiment(sentence):
    model.eval()

    input_tensor = prepare_sentence(sentence)

    with torch.no_grad():
        probability = model(input_tensor).item()

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return sentiment, probability

In [44]:
custom_sentences = [
    "I absolutely loved this product and the service was excellent.",
    "This was a terrible experience and I completely regret buying it.",
    "The product works well and I am very satisfied with my purchase."
]

In [45]:
print("=" * 70)
print("CUSTOM SENTIMENT PREDICTIONS")
print("=" * 70)

for sentence in custom_sentences:
    sentiment, probability = predict_sentiment(sentence)

    print(f"\nReview: {sentence}")
    print(f"Prediction: {sentiment}")
    print(f"Positive probability: {probability:.2%}")

CUSTOM SENTIMENT PREDICTIONS

Review: I absolutely loved this product and the service was excellent.
Prediction: Positive
Positive probability: 100.00%

Review: This was a terrible experience and I completely regret buying it.
Prediction: Negative
Positive probability: 0.16%

Review: The product works well and I am very satisfied with my purchase.
Prediction: Positive
Positive probability: 100.00%


In [46]:
def predict_with_confidence(sentence):
    sentiment, positive_probability = predict_sentiment(sentence)

    if sentiment == "Positive":
        confidence = positive_probability
    else:
        confidence = 1 - positive_probability

    return sentiment, confidence

In [47]:
print("=" * 70)
print("FINAL CUSTOMER SENTIMENT ANALYSIS")
print("=" * 70)

for number, sentence in enumerate(custom_sentences, start=1):
    sentiment, confidence = predict_with_confidence(sentence)

    print(f"\nSentence {number}: {sentence}")
    print(f"Sentiment: {sentiment}")
    print(f"Confidence: {confidence:.2%}")

FINAL CUSTOMER SENTIMENT ANALYSIS

Sentence 1: I absolutely loved this product and the service was excellent.
Sentiment: Positive
Confidence: 100.00%

Sentence 2: This was a terrible experience and I completely regret buying it.
Sentiment: Negative
Confidence: 99.84%

Sentence 3: The product works well and I am very satisfied with my purchase.
Sentiment: Positive
Confidence: 100.00%


In [48]:
loaded_model = SentimentModel(len(word_to_index))

loaded_model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Saved model loaded successfully.")

Saved model loaded successfully.
